In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import json

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
# Notebook is in: <project_root>/03_Sequences/
# So project root is one level up:
ROOT = Path.cwd().parent

FEATURE_CSV = ROOT / "02_Features" / "cadli_btcusd_1m_features_labels_2021-06-01_to_2025-12-01.csv"
OUT_DIR     = ROOT / "03_Sequences"

print("ROOT       :", ROOT)
print("FEATURE_CSV:", FEATURE_CSV)
print("OUT_DIR    :", OUT_DIR)

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load features dataframe
# ---------------------------------------------------------
df = pd.read_csv(
    FEATURE_CSV,
    parse_dates=["datetime"],
)

df = df.sort_values("datetime").set_index("datetime")

print(f"Number of rows   : {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print("Shape  :", df.shape)

df.head()


ROOT       : /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project
FEATURE_CSV: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/02_Features/cadli_btcusd_1m_features_labels_2021-06-01_to_2025-12-01.csv
OUT_DIR    : /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences
Number of rows   : 2367346
Number of columns: 34
Shape  : (2367346, 34)


,OPEN,HIGH,LOW,CLOSE,VOLUME,QUOTE_VOLUME,VOLUME_TOP_TIER,QUOTE_VOLUME_TOP_TIER,VOLUME_DIRECT,QUOTE_VOLUME_DIRECT,...,volume_ratio_15m,window_start,window_end,t_since_start_min,t_to_end_min,open_15,close_15,y_up,log_rel_to_open,rel_to_open_pct
datetime,,,,,,,,,,,,,,,,,,,,,
2021-06-01 00:15:00+00:00,37596.978146,37596.978146,37588.295217,37588.295217,386.274550,1.454586e+07,202.381039,7.610078e+06,101.682819,3.819137e+06,...,0.666775,2021-06-01 00:15:00+00:00,2021-06-01 00:30:00+00:00,0.0,15.0,37588.295217,37735.589397,1,0.000000,0.000000
2021-06-01 00:16:00+00:00,37588.295217,37602.223775,37588.295217,37602.223775,280.578555,1.055610e+07,125.918843,4.733019e+06,59.350673,2.229784e+06,...,0.495613,2021-06-01 00:15:00+00:00,2021-06-01 00:30:00+00:00,1.0,14.0,37588.295217,37735.589397,1,0.000370,0.000371
2021-06-01 00:17:00+00:00,37602.223775,37602.223775,37576.860725,37576.860725,280.033526,1.052722e+07,155.500319,5.839973e+06,55.016749,2.066074e+06,...,0.513584,2021-06-01 00:15:00+00:00,2021-06-01 00:30:00+00:00,2.0,13.0,37588.295217,37735.589397,1,-0.000304,-0.000304
2021-06-01 00:18:00+00:00,37576.860725,37576.860725,37549.917342,37549.917342,381.019556,1.430533e+07,216.922688,8.140797e+06,91.409644,3.429427e+06,...,0.834134,2021-06-01 00:15:00+00:00,2021-06-01 00:30:00+00:00,3.0,12.0,37588.295217,37735.589397,1,-0.001022,-0.001021
2021-06-01 00:19:00+00:00,37549.917342,37549.917342,37548.286482,37548.286482,276.513990,1.039699e+07,101.151345,3.803355e+06,35.871824,1.345747e+06,...,0.659693,2021-06-01 00:15:00+00:00,2021-06-01 00:30:00+00:00,4.0,11.0,37588.295217,37735.589397,1,-0.001065,-0.001064


In [2]:
# ---------------------------------------------------------
# Select features and label
# ---------------------------------------------------------

time_feature = "t_to_end_min"   # change to t_to_end_min if you used that name

feature_cols = [
    "log_ret_1m", "log_ret_5m", "log_ret_15m",
    "vol_5m", "vol_15m",
    "range_pct_1m", "range_pct_5m",
    "body_pct", "upper_wick_pct", "lower_wick_pct", "body_norm",
    "volume_ratio_5m", "volume_ratio_15m",
    "log_rel_to_open",
    time_feature,
]

label_col = "y_up"

# Sanity check: ensure all columns exist
missing = [c for c in feature_cols + [label_col] if c not in df.columns]
print("Missing columns:", missing)


Missing columns: []


In [3]:
# ---------------------------------------------------------
# Build sliding sequences
# ---------------------------------------------------------
SEQ_LEN = 120  # last 60 minutes
print("Using SEQ_LEN =", SEQ_LEN)

# Extract model input features as a NumPy array.
# Shape: (num_rows, num_features)
# Each row is one minute of feature data.
values = df[feature_cols].values

# Extract the label column as a NumPy array.
# Shape: (num_rows,)
# Each entry is the up/down label (0 or 1) for that minute.
labels = df[label_col].values

# Containers for the final sliding-window dataset.
X_seq = []  # will become (num_samples, SEQ_LEN, num_features)
y_seq = []  # will become (num_samples,)

# Build one sequence for every minute that has a full preceding SEQ_LEN window.
# For row index i:
#   - Use the SEQ_LEN rows including i as the input sequence: df[i-SEQ_LEN+1 : i+1]
#   - Use the label at row i as the target for that sequence.
for i in range(SEQ_LEN - 1, len(df)):
    
    # Extract the past SEQ_LEN minutes of features.
    # Shape: (SEQ_LEN, num_features)
    seq = values[i-SEQ_LEN+1 : i+1]  # (SEQ_LEN, num_features)

    # Append the sequence to the dataset.
    X_seq.append(seq)

    # Append the label for this time index (prediction target).
    y_seq.append(labels[i])     # label at time i

# Convert the collected lists into NumPy arrays for model training.
X_seq = np.array(X_seq) # (num_samples, SEQ_LEN, num_features)
y_seq = np.array(y_seq, dtype=int)  # (num_samples,)

# Check that each sequence ends with the same row used for the label
for k in [0, 1000, 50000]:  # any random samples
    if k < len(X_seq):
        print("Seq last row equals df row for label?",
              np.allclose(X_seq[k][-1], values[(SEQ_LEN - 1) + k]))

print("X_seq shape:", X_seq.shape)  # (N, SEQ_LEN, num_features)
print("y_seq shape:", y_seq.shape)  # (N,)

# Quick check of the class distribution.
# This is the proportion of 'up' labels in the dataset.
print("Positive label ratio:", y_seq.mean())


Using SEQ_LEN = 120
Seq last row equals df row for label? True
Seq last row equals df row for label? True
Seq last row equals df row for label? True
X_seq shape: (2367227, 120, 15)
y_seq shape: (2367227,)
Positive label ratio: 0.5002118512504293


In [4]:
# ---------------------------------------------------------
# Time-based train/test split
# ---------------------------------------------------------

# We use a strict chronological split, NOT a random split.
# This is critical in time-series prediction to avoid leaking
# future information into the training set.
train_ratio = 0.8

# Index where we split train/test by time.
split_idx = int(len(X_seq) * train_ratio)

# Training set: all sequences up to split_idx
X_train = X_seq[:split_idx]
y_train = y_seq[:split_idx]

# Test set: all sequences after split_idx
X_test  = X_seq[split_idx:]
y_test  = y_seq[split_idx:]

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_test :", X_test.shape,  " y_test :", y_test.shape)


X_train: (1893781, 120, 15)  y_train: (1893781,)
X_test : (473446, 120, 15)  y_test : (473446,)


In [5]:
# ---------------------------------------------------------
# Save arrays and metadata
# ---------------------------------------------------------
from pathlib import Path

OUT_DIR = OUT_DIR / f"seq{SEQ_LEN}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(OUT_DIR / "X_train.npy", X_train)
np.save(OUT_DIR / "y_train.npy", y_train)
np.save(OUT_DIR / "X_test.npy",  X_test)
np.save(OUT_DIR / "y_test.npy",  y_test)

meta = {
    "seq_len": SEQ_LEN,
    "feature_cols": feature_cols,
    "label_col": label_col,
    "train_ratio": train_ratio,
    "num_train": int(X_train.shape[0]),
    "num_test": int(X_test.shape[0]),
}

with open(OUT_DIR / "meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Saved to:", OUT_DIR)
print("Meta:", meta)


Saved to: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/seq120
Meta: {'seq_len': 120, 'feature_cols': ['log_ret_1m', 'log_ret_5m', 'log_ret_15m', 'vol_5m', 'vol_15m', 'range_pct_1m', 'range_pct_5m', 'body_pct', 'upper_wick_pct', 'lower_wick_pct', 'body_norm', 'volume_ratio_5m', 'volume_ratio_15m', 'log_rel_to_open', 't_to_end_min'], 'label_col': 'y_up', 'train_ratio': 0.8, 'num_train': 1893781, 'num_test': 473446}
